# 01 — Data Exploration

Download and inspect the four SNAP BioSNAP datasets. Audit ID formats, measure overlaps, and generate exploratory plots (E1–E3).

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter

from src.data_loading import load_all, download_all
from src.plotting import plot_degree_distribution, plot_id_overlap
from src.utils import set_seed, BLOG_ASSETS

set_seed(42)
%matplotlib inline

## 1. Download & Load All Datasets

In [ ]:
download_all()
dfs = load_all()

ddi = dfs['ChCh-Miner']
dtg = dfs['ChG-Miner']
dga = dfs['DG-AssocMiner']
dda = dfs['DCh-Miner']

print(f"DDI (Drug-Drug):     {len(ddi):>8,} edges")
print(f"DTG (Drug-Gene):     {len(dtg):>8,} edges")
print(f"DGA (Disease-Gene):  {len(dga):>8,} edges")
print(f"DDA (Disease-Drug):  {len(dda):>8,} edges")

## 2. Inspect Raw Formats & ID Schemes

In [ ]:
for name, df in dfs.items():
    print(f"\n{'='*40}")
    print(f"{name}")
    print(f"{'='*40}")
    print(f"Shape: {df.shape}")
    print(f"Columns: {list(df.columns)}")
    print(f"\nSample rows:")
    display(df.head(3))
    print(f"\nID examples per column:")
    for col in df.columns:
        examples = df[col].unique()[:5]
        print(f"  {col}: {list(examples)}")

## 3. Node Counts & Unique IDs

In [ ]:
ddi_drugs = set(ddi['drug1'].unique()) | set(ddi['drug2'].unique())
dtg_drugs = set(dtg['drug'].unique())
dtg_genes = set(dtg['gene'].unique())
dda_drugs = set(dda['drug'].unique())
dda_diseases = set(dda['disease'].unique())
dga_diseases = set(dga['disease'].unique())
dga_genes = set(dga['gene'].unique())

print(f"DDI drugs (ChCh-Miner):       {len(ddi_drugs):>6,}")
print(f"DTG drugs (ChG-Miner):        {len(dtg_drugs):>6,}")
print(f"DTG genes (ChG-Miner):        {len(dtg_genes):>6,}  (UniProt IDs)")
print(f"DDA drugs (DCh-Miner):        {len(dda_drugs):>6,}")
print(f"DDA diseases (DCh-Miner):     {len(dda_diseases):>6,}  (MESH IDs)")
print(f"DGA diseases (DG-AssocMiner): {len(dga_diseases):>6,}  (CUI IDs)")
print(f"DGA genes (DG-AssocMiner):    {len(dga_genes):>6,}  (ENTREZ IDs)")

## 4. Drug ID Overlap Across Datasets

In [ ]:
overlap_ddi_dtg = ddi_drugs & dtg_drugs
overlap_ddi_dda = ddi_drugs & dda_drugs
overlap_all3 = ddi_drugs & dtg_drugs & dda_drugs

print(f"Drugs in DDI ∩ DTG:          {len(overlap_ddi_dtg):>6,}  ({100*len(overlap_ddi_dtg)/len(ddi_drugs):.1f}% of DDI drugs)")
print(f"Drugs in DDI ∩ DDA:          {len(overlap_ddi_dda):>6,}  ({100*len(overlap_ddi_dda)/len(ddi_drugs):.1f}% of DDI drugs)")
print(f"Drugs in DDI ∩ DTG ∩ DDA:    {len(overlap_all3):>6,}  ({100*len(overlap_all3)/len(ddi_drugs):.1f}% of DDI drugs)")
print(f"DDI drugs NOT in DTG or DDA: {len(ddi_drugs - dtg_drugs - dda_drugs):>6,}")

overlap_counts = {
    'DDI only': len(ddi_drugs - dtg_drugs - dda_drugs),
    'DDI ∩ DTG only': len((ddi_drugs & dtg_drugs) - dda_drugs),
    'DDI ∩ DDA only': len((ddi_drugs & dda_drugs) - dtg_drugs),
    'DDI ∩ DTG ∩ DDA': len(overlap_all3),
}
fig = plot_id_overlap(overlap_counts, title='DDI Drug Coverage by Auxiliary Datasets')
plt.show()

## 5. Gene ID Overlap (UniProt vs Entrez)

ChG-Miner uses UniProt gene IDs while DG-AssocMiner uses Entrez Gene IDs. These are different identifier systems, so we expect zero direct overlap. Both are treated as "gene" nodes but in separate ID pools.

In [ ]:
gene_overlap = dtg_genes & dga_genes
print(f"Gene overlap (UniProt ∩ Entrez): {len(gene_overlap)}")
print(f"Total unique gene IDs: {len(dtg_genes | dga_genes):,}")
print(f"  UniProt (from ChG-Miner):      {len(dtg_genes):,}")
print(f"  Entrez  (from DG-AssocMiner):   {len(dga_genes):,}")
print()
print("Note: Gene nodes won't bridge drug-gene and disease-gene directly.")
print("The disease-drug link (DCh-Miner) provides the primary bridge.")

## 6. E1: Drug Degree Distribution in DDI Network

In [ ]:
degree_counts = Counter()
degree_counts.update(ddi['drug1'].values)
degree_counts.update(ddi['drug2'].values)
degrees = list(degree_counts.values())

print(f"Degree stats:")
print(f"  Min:    {min(degrees)}")
print(f"  Max:    {max(degrees)}")
print(f"  Mean:   {np.mean(degrees):.1f}")
print(f"  Median: {np.median(degrees):.1f}")

fig = plot_degree_distribution(degrees, title='Drug Degree Distribution (DDI Network)')
plt.show()

## 7. E3: Gene Degree Distribution in Drug-Gene Network

In [ ]:
gene_degree = Counter(dtg['gene'].values)
gene_degrees = list(gene_degree.values())

print(f"Gene degree stats (drug-gene network):")
print(f"  Min:    {min(gene_degrees)}")
print(f"  Max:    {max(gene_degrees)}")
print(f"  Mean:   {np.mean(gene_degrees):.1f}")
print(f"  Median: {np.median(gene_degrees):.1f}")

fig = plot_degree_distribution(gene_degrees, title='Gene Degree Distribution (Drug-Gene Network)')
plt.show()

## 8. Disease-Drug Statistics

In [ ]:
dda_filtered = dda[dda['drug'].isin(ddi_drugs)]
print(f"DCh-Miner edges total:               {len(dda):>8,}")
print(f"DCh-Miner edges with DDI drugs only:  {len(dda_filtered):>8,}")
print(f"Diseases linked to DDI drugs:          {dda_filtered['disease'].nunique():>8,}")
print(f"DDI drugs with disease associations:   {dda_filtered['drug'].nunique():>8,}")

## 9. Summary Statistics Table

In [ ]:
summary = pd.DataFrame({
    'Dataset': ['ChCh-Miner (DDI)', 'ChG-Miner (DTG)', 'DG-AssocMiner (DGA)', 'DCh-Miner (DDA)'],
    'Edges': [len(ddi), len(dtg), len(dga), len(dda)],
    'Node Type 1': ['Drug', 'Drug', 'Disease', 'Disease'],
    'Unique Type 1': [len(ddi_drugs), len(dtg_drugs), len(dga_diseases), len(dda_diseases)],
    'Node Type 2': ['Drug', 'Gene (UniProt)', 'Gene (Entrez)', 'Drug'],
    'Unique Type 2': ['-', len(dtg_genes), len(dga_genes), len(dda_drugs)],
})
display(summary)

print("\n=== Key Takeaways ===")
print(f"Core DDI graph: {len(ddi_drugs)} drugs, {len(ddi)} edges")
print(f"Drug-gene targets cover {100*len(overlap_ddi_dtg)/len(ddi_drugs):.0f}% of DDI drugs")
print(f"Disease-drug assocs cover {100*len(overlap_ddi_dda)/len(ddi_drugs):.0f}% of DDI drugs")
print(f"Gene ID systems do NOT overlap (UniProt vs Entrez) — noted as limitation")